# 5. A monotone decision tree

The data dictionary states how every feature should affect the risk, i.e.,
its *monotonicity constraint*.
The original tree does not obey these constraints (notebook 4):
some of its explanations say that, e.g., more credit checks would have helped.
scikit-learn can train trees that respect them;
this notebook compares such a tree with the original one.

The constraints of the ten model features (1 indicates that more means a higher
risk, -1 a lower risk and 0 no constraint):

In [1]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
from fico_xml import data, evaluation, model, preprocessing
from fico_xml.explain import explain

raw = data.load_raw()
clean = preprocessing.clean(raw)
pd.Series(model.MONOTONICITY, name='constraint')

NumTotalTrades                        0
NumSatisfactoryTrades                -1
NumTrades60Ever2DerogPubRec           1
NumTrades90Ever2DerogPubRec           1
NumTradesOpeninLast12M                1
NumRevolvingTradesWBalance            0
NumInstallTradesWBalance              0
NumBank2NatlTradesWHighUtilization    1
NumInqLast6Mexcl7days                 1
NumInqLast7DCalc                      1
Name: constraint, dtype: int64

## Training

The same settings as the original tree (depth 7, at least 100 applicants per
leaf), with the constraints:

In [2]:
original = model.train(clean)
monotone = model.train(clean, monotone=True)
pd.DataFrame({'leaves': [original.classifier.get_n_leaves(), monotone.classifier.get_n_leaves()],
              'depth': [original.classifier.get_depth(), monotone.classifier.get_depth()]},
             index=['original', 'monotone'])

,leaves,depth
original,42,7
monotone,51,7


Its cross-validated accuracy compared with the original tree:

In [3]:
from sklearn.tree import DecisionTreeClassifier

X, y = model.model_data(clean)
constraints = [-model.MONOTONICITY[f] for f in model.MODEL_FEATURES]  # for P(Good)
pd.DataFrame({
    'original': evaluation.cross_validate(DecisionTreeClassifier(random_state=0, **model.ORIGINAL_PARAMETERS), X, y).mean(),
    'monotone': evaluation.cross_validate(DecisionTreeClassifier(random_state=0, monotonic_cst=constraints,
                                                                 **model.ORIGINAL_PARAMETERS), X, y).mean()}).round(3)

,original,monotone
train accuracy,0.682,0.646
train F1 (Good),0.673,0.663
test accuracy,0.659,0.640
test F1 (Good),0.647,0.659


The constraints cost about two points of test accuracy (0.64 instead of 0.66),
while the F1 score of `Good` is slightly higher.

The features it relies on:

In [4]:
pd.DataFrame({'original': original.feature_importance(), 'monotone': monotone.feature_importance()}).round(3)

,original,monotone
NumBank2NatlTradesWHighUtilization,0.477,0.505
NumInqLast6Mexcl7days,0.044,0.017
NumInqLast7DCalc,0.000,0.000
NumInstallTradesWBalance,0.011,0.053
NumRevolvingTradesWBalance,0.029,0.133
NumSatisfactoryTrades,0.181,0.203
NumTotalTrades,0.024,0.084
NumTrades60Ever2DerogPubRec,0.189,0.000
NumTrades90Ever2DerogPubRec,0.000,0.000
NumTradesOpeninLast12M,0.044,0.005


## Explanations

The applicants of notebook 4, explained by the monotone tree:

In [5]:
for i in (34, 2, 7890):
    print('Applicant', i)
    print(explain(monotone, raw.loc[i], warn=False).text(limit=2), end='\n\n')

Applicant 34
The application is declined.
The application would be accepted had the number of instalment loans with a balance been at most 2 (instead of 3).
The application would be accepted had there been no bank or national credit-card accounts with high utilisation (instead of 1).
The application would still be declined had the number of credit accounts in good standing been at most 19 or between 28 and 32 (instead of 23).
The application would still be declined had the number of revolving accounts (credit cards, credit lines) with a balance been at least 8 (instead of 5).

Applicant 2
The application is declined.
The application would be accepted had there been no bank or national credit-card accounts with high utilisation (instead of 1).
The application would be accepted had the number of credit accounts in good standing been at least 20 (instead of 9).
The application would still be declined had the number of credit checks in the last 6 months (excluding the last 7 days) been at 

The application is accepted.
The application would be declined had the number of bank or national credit-card accounts with high utilisation been at least 1 (instead of 0) and the number of revolving accounts (credit cards, credit lines) with a balance been between 6 and 7 (instead of 2).
The application would be declined had the number of bank or national credit-card accounts with high utilisation been at least 1 (instead of 0) and the number of revolving accounts (credit cards, credit lines) with a balance been exactly 8 (instead of 2).
The application would still be accepted had the number of revolving accounts (credit cards, credit lines) with a balance been at least 3 (instead of 2).
The application would still be accepted had the number of credit accounts in good standing been at most 14 or at least 25 (instead of 20).



The explanations change:
for the first applicant, whose decision depended on past delinquency under
the original tree, the monotone tree points to paying off an instalment loan or
a highly utilised credit card instead.

## Monotonicity of the explanations

A counterfactual leads to a particular leaf and the leaf's region may happen to
require a change against a constraint, so even a monotone tree can produce such
counterfactuals.
For a monotone tree, however, these changes are never needed:
undoing them keeps the reversed decision.
A random sample of 300 applicants:

In [6]:
sample = X.sample(300, random_state=0).index
rows = []
for name, tree in (('original', original), ('monotone', monotone)):
    counterfactuals, needed = 0, 0
    contradicting = 0
    for i in sample:
        e = explain(tree, raw.loc[i], warn=False)
        for cf in e.contrastive:
            counterfactuals += 1
            if cf.contradicts_monotonicity:
                contradicting += 1
                undone = {f: v for f, v in cf.values.items() if f not in cf.contradicts_monotonicity}
                still_reversed = tree.predict(pd.DataFrame([dict(e.values, **undone)]))[0] != e.prediction
                needed += not still_reversed
    rows.append({'tree': name, 'counterfactuals': counterfactuals, 'contradicting monotonicity': contradicting,
                 '... where the contradicting change is needed': needed})
pd.DataFrame(rows).set_index('tree')

,counterfactuals,contradicting monotonicity,... where the contradicting change is needed
tree,,,
original,5852,2451,116
monotone,7452,2218,0


With the monotone tree no explanation *relies* on a change against the
constraints:
dropping such changes (which `explain` flags) gives an explanation that agrees
with the data dictionary.
In the original tree, some explanations need them.